# 02 — Modelado y decisión

Dos objetos distintos, que se eligen con criterios distintos:

- un **modelo**, que produce una probabilidad de baja;
- una **política**, que convierte esa probabilidad en la decisión de contactar
  o no contactar.

Confundirlos es el error más común en este tipo de trabajo. Un umbral de 0.5 no
es una propiedad del modelo: es una convención heredada de problemas
balanceados, sin ningún contenido económico acá.

## Protocolo

1. `churn_train.csv` se usa para todo el desarrollo.
2. Selección de modelo por validación cruzada estratificada de 5 folds sobre
   train, optimizando PR-AUC.
3. El umbral se fija con predicciones **fuera de muestra** (`cross_val_predict`)
   sobre train.
4. `churn_test.csv` se evalúa **una sola vez**, al final, con modelo y umbral
   ya congelados.

El paso 3 es el que corrige el problema principal de la versión original: si el
umbral se busca sobre el mismo conjunto donde después se reporta performance, es
un hiperparámetro ajustado al test y las métricas quedan infladas.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src import data as D
from src import evaluate as E
from src import interpret as I
from src import segment as S
from src.features import add_engineered_features, build_preprocessor
from src.models import build_model_zoo

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

In [ ]:
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
)

train = add_engineered_features(D.load_split("train"))
test = add_engineered_features(D.load_split("test"))

X_train, y_train = D.split_X_y(train)
X_test, y_test = D.split_X_y(test)

cv = StratifiedKFold(n_splits=C.N_FOLDS, shuffle=True, random_state=C.SEED)
print(f"Desarrollo: {len(X_train):,} filas   Holdout: {len(X_test):,} filas")

## 1. Por qué el preprocesamiento va dentro del pipeline

Si el `StandardScaler` y el `OneHotEncoder` se ajustaran una sola vez sobre
todos los datos y después se pasaran las matrices ya transformadas a la
validación cruzada, cada fold de validación habría contribuido a la media y el
desvío del escalado. El sesgo es chico pero sistemático, y desaparece gratis
metiendo el preprocesador dentro del `Pipeline`.

Todos los modelos de `src/models.py` están construidos así.

In [ ]:
zoo = build_model_zoo(engineered=True)

pd.DataFrame([
    {"modelo": s.name,
     "escalado": "sí" if s.needs_scaling else "no",
     "combinaciones": int(np.prod([len(v) for v in s.param_grid.values()]) or 1),
     "rol": s.nota}
    for s in zoo
]).set_index("modelo")

Cuatro familias con supuestos distintos:

- **Lineales regularizados** (L2, L1, elastic net): asumen que el log-odds es
  lineal en los predictores. Son el baseline interpretable.
- **KNN**: no paramétrico, sin forma funcional impuesta, pero sensible a la
  escala y a la dimensión.
- **Árbol con poda de coste-complejidad**: captura interacciones y se puede
  leer como reglas.
- **Ensambles**: bagging (Random Forest, Extra Trees) y boosting
  (Hist Gradient Boosting).

Sobre el desbalance: se usa `class_weight="balanced"`, que reponderar la clase
minoritaria dentro de la función de pérdida, en vez de remuestreo sintético
(SMOTE, ADASYN). Con 16% de positivos y 8.100 filas la clase rara no es tan rara
como para necesitar ejemplos artificiales, y reponderar no distorsiona la
distribución conjunta de las variables. El desbalance real se maneja en el
umbral, no cambiando los datos.

## 2. Validación cruzada

La celda siguiente tarda varios minutos. Los resultados ya calculados están en
`reports/results/02_comparacion_modelos.csv`.

In [ ]:
resultados, ajustados = [], {}

for spec in zoo:
    busqueda = GridSearchCV(spec.estimator, spec.param_grid,
                            scoring="average_precision", cv=cv, n_jobs=-1)
    busqueda.fit(X_train, y_train)
    ajustados[spec.name] = busqueda.best_estimator_
    i = busqueda.best_index_
    resultados.append({
        "modelo": spec.name,
        "pr_auc_cv": busqueda.cv_results_["mean_test_score"][i],
        "pr_auc_std": busqueda.cv_results_["std_test_score"][i],
        "mejores_params": busqueda.best_params_,
    })
    print(f"{spec.name:<26} PR-AUC = {resultados[-1]['pr_auc_cv']:.4f}")

comparacion = pd.DataFrame(resultados).sort_values("pr_auc_cv", ascending=False)
comparacion.set_index("modelo").round(4)

El orden es nítido: boosting (0.967) > bagging (0.943 y 0.923) > árbol
único (0.854) > KNN (0.833) > lineales (0.795).

La brecha de 17 puntos entre el boosting y las logísticas confirma lo que
anticipaba el notebook 01: **el poder predictivo está en las interacciones**.
Ninguna correlación bivariada superaba 0.4 y sin embargo el ordenamiento es
casi perfecto.

Las tres logísticas empatan en la tercera cifra decimal. Eso significa que la
regularización no está haciendo trabajo: con 24 predictores y 8.100
observaciones no hay sobreajuste que corregir. El Lasso no descarta variables
porque no le sobran.

## 3. Ablación: ¿sirven las variables derivadas?

La pregunta honesta sobre cualquier feature engineering es si aporta algo. Se
compara el mismo modelo con y sin las variables construidas.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

for con_derivadas in (False, True):
    df = D.load_split("train")
    if con_derivadas:
        df = add_engineered_features(df)
    X, y = D.split_X_y(df)
    pipe = Pipeline([
        ("pre", build_preprocessor(engineered=con_derivadas, scale=False)),
        ("clf", HistGradientBoostingClassifier(random_state=C.SEED)),
    ])
    s = cross_val_score(pipe, X, y, scoring="average_precision", cv=cv)
    print(f"variables derivadas = {str(con_derivadas):<5}  "
          f"PR-AUC = {s.mean():.4f} ± {s.std():.4f}")

**0.9679 sin ellas, 0.9678 con ellas.** Las variables derivadas no
aportan nada.

Tiene sentido: el boosting parte recursivamente el espacio de predictores, y un
ratio como `monto / cantidad` es reconstruible mediante cortes sucesivos en sus
dos componentes. El feature engineering ayuda cuando el modelo no puede expresar
la transformación por sí mismo, que es el caso de los lineales, no el de los
ensambles de árboles.

Se dejan en el repositorio porque mejoran la interpretabilidad de la logística,
y porque el resultado negativo es parte del análisis.

## 4. Del score a la decisión

El modelo devuelve probabilidades. Falta decidir a partir de qué valor se
contacta a un cliente. Se evalúan cuatro criterios sobre predicciones fuera de
muestra:

- **F1 máximo**: criterio estadístico, equilibra precisión y recall sin
  referencia a costos.
- **Capacidad**: contacta al 30% más riesgoso, el techo operativo del equipo
  de retención.
- **Costo mínimo**: minimiza el costo esperado. Perder un cliente cuesta 100;
  contactar a uno que se iba a quedar, 15. Son supuestos ilustrativos: lo que
  importa es la relación entre ambos, no la magnitud.
- **Convención 0.5**: la referencia a batir.

In [ ]:
mejor_nombre = comparacion.iloc[0]["modelo"]
mejor = ajustados[mejor_nombre]

oof = cross_val_predict(mejor, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
politicas = E.compare_threshold_policies(y_train, oof)
politicas.round(4)

Comparar las filas de capacidad y costo mínimo:

| criterio | cartera contactada | precisión | recall | costo |
|---|---|---|---|---|
| Capacidad (30%) | 30.0% | 0.53 | 0.995 | 17.740 |
| Costo mínimo | 20.0% | 0.78 | 0.972 | 8.880 |

Contactar al 30% en vez del 20% gana 2,2 puntos de recall y **duplica el
costo**. La capacidad del equipo es un techo, no un objetivo: agotarla porque
existe multiplica los falsos positivos sin ganar casi nada.

Por eso el umbral congelado usa el criterio de costo, con `THRESHOLD_POLICY` en
`src/config.py`. El 20% que resulta está holgadamente por debajo del techo del
30%, así que la restricción operativa no es activa.

In [ ]:
umbral = E.freeze_threshold(y_train, oof, C.THRESHOLD_POLICY)
print(f"Umbral congelado ({C.THRESHOLD_POLICY}): {umbral:.4f}")
print(f"Contacta al {(oof >= umbral).mean():.1%} de la cartera "
      f"(techo operativo: {C.CAPACITY:.0%})")

E.decile_table(y_train, oof)

La tabla de deciles se calcula sobre predicciones **out-of-fold**, no
sobre predicciones en entrenamiento. Ese es el punto que corrige el error de la
versión original, donde el modelo se reajustaba sobre todos los datos y después
se scoreaba ese mismo conjunto.

El primer decil concentra el 61% de las bajas con un lift de 6,1. Los primeros
dos deciles acumulan el 97%.

## 5. Interpretabilidad

Dos preguntas distintas que requieren herramientas distintas.

**Cuánto aporta cada variable**: importancia por permutación, calculada fuera
de muestra. Se prefiere al `feature_importances_` de los árboles, que se
calcula en entrenamiento y sobrevalora a las variables de alta cardinalidad.

**En qué dirección empuja**: odds ratios de la logística L1.

Sobre esto último hay que ser explícito. La versión original de este trabajo
ajustaba `LassoCV` —que es **regresión lineal** con penalización L1— sobre un
target binario, y después exponenciaba los coeficientes para presentarlos como
odds ratios. Eso no es válido: exponenciar un coeficiente de mínimos cuadrados
no produce un odds ratio, produce un número sin interpretación. Los odds ratios
requieren un modelo logístico. `src/interpret.py` levanta un `TypeError` si se
le pasa un estimador sin `coef_` logístico.

In [ ]:
imp = I.permutation_importances(mejor, X_train, y_train, n_repeats=5)
imp.head(10).round(4)

In [ ]:
ors = I.odds_ratios(ajustados["logistica_l1"], top=12)
ors[["etiqueta", "coeficiente", "odds_ratio", "efecto"]].round(3)

Las dos listas coinciden en lo esencial —cantidad y monto de
transacciones dominan— pero difieren en el orden, porque miden cosas distintas.
La logística atribuye un odds ratio alto al ticket promedio; el boosting le da
un tercio de la importancia que le da a la cantidad de transacciones. La
diferencia viene de que la logística no puede representar la interacción entre
monto y cantidad, y la comprime en el cociente.

Un detalle a no sobreinterpretar: el género masculino aparece con odds ratio
0.44. En el notebook 01 vimos que la diferencia por género no sobrevive al
intervalo de confianza. Un coeficiente ajustado por otras variables puede ser
estadísticamente distinguible sin que la variable tenga poder predictivo real:
en la importancia por permutación, el género no aparece entre las diez
primeras.

## 6. Holdout: una sola evaluación

Modelo y umbral congelados. Esta celda se ejecuta una vez.

In [ ]:
mejor.fit(X_train, y_train)
proba_test = mejor.predict_proba(X_test)[:, 1]

libres = E.threshold_free_metrics(y_test, proba_test)
decision = E.metrics_at_threshold(y_test, proba_test, umbral)

print("Métricas independientes del umbral")
for k, v in libres.items():
    print(f"  {k:>14}: {v:.4f}")

print(f"\nDecisión al umbral congelado ({umbral:.4f})")
for k in ["precision", "recall", "f1", "accuracy", "cartera_contactada"]:
    print(f"  {k:>18}: {decision[k]:.4f}")
print(f"\n  Matriz de confusión: TP={decision['TP']} FP={decision['FP']} "
      f"FN={decision['FN']} TN={decision['TN']}")

ROC-AUC 0.992, PR-AUC 0.965, Brier 0.023. Al umbral congelado:
precisión 0.78, recall 0.95, contactando al 19,5% de la cartera.

Las métricas del holdout coinciden con las de validación cruzada, lo que indica
que no hubo sobreajuste en la selección.

**Advertencia necesaria.** Un ROC-AUC de 0.99 no es normal y no debe leerse
como mérito del modelo. Se auditó: no hay filas duplicadas dentro de cada
partición, no hay solapamiento entre train y test, y ninguna variable separa
por sí sola (el mejor AUC univariado es 0.796). El dataset —el
*Credit Card Customers* de Kaggle— es sintético o fuertemente depurado, y
resulta casi separable para ensambles de árboles. Sobre datos bancarios reales,
con clases del orden del 2% y ruido de medición, cualquier resultado de esta
magnitud sería motivo para buscar el error antes que para celebrar.

In [ ]:
E.decile_table(y_test, proba_test)